# AI for IB — Colab benchmark and QLoRA

This notebook keeps private training/evaluation files in your Google Drive and public code in GitHub. Run the cells in order. Start with a **GPU runtime**.

Do not paste ManageBac passwords, cookies, Supabase secret keys, or licensed PDFs into notebook cells.

In [ ]:
import os, subprocess, pathlib
REPO = pathlib.Path('/content/Ai-for-ib')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/DH4410/Ai-for-ib.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())

In [ ]:
!pip install --upgrade --no-cache-dir -r training/requirements-colab.txt


## 1. Check the assigned GPU
Paste the JSON output into ChatGPT if you want the model/batch settings tuned to this exact runtime.

In [ ]:
!python training/colab_preflight.py


## 2. Public smoke benchmark
This uses the repository's 12 synthetic cases (Physics/Chemistry/Mathematics × Learn/Practice/Mark/Revise). It contains no private source material.

In [ ]:
!python training/benchmark_models.py \
  --eval training/eval.example.jsonl \
  --models 'Qwen/Qwen3-4B' \
  --max-cases 12 \
  --output training/outputs/public-smoke-benchmark.json


In [ ]:
import json
report = json.load(open('training/outputs/public-smoke-benchmark.json', encoding='utf-8'))
print(json.dumps(report.get('ranking', []), indent=2))

## 3. Mount Drive for private files/checkpoints
Create a Drive folder such as `MyDrive/ai-for-ib/private/`. Keep real `train.jsonl` and `eval.jsonl` there, not in Git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PRIVATE_DIR = '/content/drive/MyDrive/ai-for-ib/private'
CHECKPOINT_DIR = '/content/drive/MyDrive/ai-for-ib/checkpoints'
os.makedirs(PRIVATE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(PRIVATE_DIR)

## 4. Choose the base model only after benchmarking
Start with Qwen3-4B unless the private benchmark shows another candidate is clearly better. If Phi-4-mini wins, set `TRUST_REMOTE_CODE = True`.

In [ ]:
BASE_MODEL = 'Qwen/Qwen3-4B'
TRUST_REMOTE_CODE = False
TRAIN_JSONL = f'{PRIVATE_DIR}/train.jsonl'
EVAL_JSONL = f'{PRIVATE_DIR}/eval.jsonl'
OUTPUT_DIR = f'{CHECKPOINT_DIR}/Dima-IB-Tutor-v1'
print(BASE_MODEL, TRAIN_JSONL, EVAL_JSONL, OUTPUT_DIR)

## 5. Private base-model benchmark
Run this **before** fine-tuning. The report stays in the ignored `training/outputs/` path for this Colab session.

In [ ]:
assert os.path.isfile(EVAL_JSONL), 'Upload eval.jsonl to your private Drive folder first.'
!python training/benchmark_models.py --eval "$EVAL_JSONL" --output training/outputs/private-base-benchmark.json


## 6. QLoRA fine-tune
Only run after the private train/eval split has been validated and the base-model benchmark has been reviewed.

In [ ]:
assert os.path.isfile(TRAIN_JSONL), 'Upload train.jsonl to your private Drive folder first.'
trust_flag = '--trust-remote-code' if TRUST_REMOTE_CODE else ''
cmd = f'''python training/colab_train.py \
  --base-model "{BASE_MODEL}" \
  --train "{TRAIN_JSONL}" \
  --eval "{EVAL_JSONL}" \
  --output-dir "{OUTPUT_DIR}" \
  --epochs 2 --learning-rate 1e-4 \
  --batch-size 1 --gradient-accumulation 16 \
  --max-length 2048 {trust_flag}'''
print(cmd)
subprocess.run(cmd, shell=True, check=True)

## 7. Adapter smoke test
This verifies that the saved LoRA loads and produces a normal tutoring response. It is not the final quality evaluation.

In [ ]:
trust_flag = '--trust-remote-code' if TRUST_REMOTE_CODE else ''
cmd = f'python training/smoke_adapter.py --base-model "{BASE_MODEL}" --adapter "{OUTPUT_DIR}" {trust_flag}'
subprocess.run(cmd, shell=True, check=True)

## 8. Promotion rule
Do **not** deploy the adapter just because training completed. Compare it with the base model on the same held-out private evaluation set and manually inspect correctness, IB appropriateness, hint quality, marking precision, hallucinations and verbosity.